# Clase 3 — Regresión y clasificación con Keras

## Objetivos de aprendizaje
- Construir tu primer modelo de deep learning usando la librería Keras.
- Entrenar una red neuronal de **regresión** que predice la resistencia del
  concreto a partir de sus ingredientes.
- Entrenar una red neuronal de **clasificación** que reconoce dígitos
  escritos a mano (MNIST) usando una arquitectura densa.
- Entender por qué la capa de salida y la función de pérdida cambian entre
  un problema de regresión y uno de clasificación.
- Comparar el flujo de trabajo de Keras (`Sequential` → `compile` → `fit`)
  con el backpropagation manual que programaste en la clase anterior.

## Qué vamos a construir
1. Un modelo de regresión con dos capas ocultas de 50 nodos que predice la
   resistencia del concreto (`concrete_data.csv`, dataset público de IBM).
2. Un modelo de clasificación con una red densa que reconoce dígitos del 0
   al 9 sobre el dataset MNIST, incluido directamente en Keras.

Todo el código de esta clase corre en Google Colab sin necesidad de
credenciales ni tokens: el dataset del concreto se descarga por URL pública
y MNIST viene incluido en `keras.datasets`.

Keras y TensorFlow ya vienen preinstalados en Colab, así que no hace falta
ningún `!pip install` para esta clase.

In [ ]:
# Verificamos la versión de Keras/TensorFlow disponible en el entorno.
# (En Colab ya vienen preinstalados; si ejecutas esto localmente y falla,
# instala con: pip install tensorflow)
import keras
print("Keras:", keras.__version__)

## Parte 1 — Regresión: resistencia del concreto

**Recordatorio teórico (capítulo 5 del manual):** en un problema de
**regresión** la red predice un número continuo. El flujo con Keras es
siempre el mismo: cargar y limpiar los datos → separar predictores y
target → normalizar (z-score) → definir el modelo con `Sequential` →
compilar → `fit`.

El dataset (`concrete_data.csv`) describe casi 1000 muestras de concreto:
cemento, escoria de alto horno, ceniza volante, agua, superplastificante,
agregado grueso, agregado fino y edad de la mezcla. La columna objetivo es
`Strength`, la resistencia a la compresión en MPa.

In [ ]:
import pandas as pd
import numpy as np

# Cargamos el dataset directamente desde el bucket S3 público de IBM.
# Es la misma URL usada en el material original del curso; no requiere
# credenciales ni token de ningún tipo.
filepath = 'https://s3-api.us-geo.objectstorage.softlayer.net/cf-courses-data/CognitiveClass/DL0101EN/labs/data/concrete_data.csv'
concrete_data = pd.read_csv(filepath)

print(concrete_data.shape)
concrete_data.head()

In [ ]:
# Revisamos que no haya valores nulos: si el dataset estuviera sucio,
# este sería el momento de limpiarlo antes de seguir.
concrete_data.isnull().sum()

In [ ]:
# Separamos predictores (todas las columnas menos "Strength") del target.
concrete_data_columns = concrete_data.columns
predictors = concrete_data[concrete_data_columns[concrete_data_columns != 'Strength']]
target = concrete_data['Strength']

# Normalización z-score: (x - media) / desviación estándar.
predictors_norm = (predictors - predictors.mean()) / predictors.std()

n_cols = predictors_norm.shape[1]  # número de columnas de entrada
print("Número de predictores:", n_cols)
predictors_norm.head()

**Arquitectura del modelo de regresión** (idéntica a la del material
original de IBM): capa de entrada con `n_cols` valores, dos capas ocultas
`Dense` de 50 nodos con activación ReLU, y una capa de salida de un solo
nodo **sin activación** (porque queremos un número real sin restricciones
de rango). Compilamos con el optimizador `adam` y pérdida
`mean_squared_error`, la combinación estándar para regresión.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense, Input

def regression_model():
    model = Sequential()
    model.add(Input(shape=(n_cols,)))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(50, activation='relu'))
    model.add(Dense(1))  # salida: un número, sin activación

    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

model = regression_model()
model.summary()

In [ ]:
# Entrenamos y validamos al mismo tiempo con fit(): 30% de los datos para
# validación, 100 épocas. En Colab esto tarda solo unos segundos porque el
# dataset es pequeño (~1000 filas).
history = model.fit(predictors_norm, target, validation_split=0.3, epochs=100, verbose=2)

In [ ]:
# Comparamos el error final de entrenamiento contra el de validación.
print("loss final (entrenamiento):", history.history['loss'][-1])
print("val_loss final (validación):", history.history['val_loss'][-1])

## Parte 2 — Clasificación: dígitos escritos a mano (MNIST)

**Recordatorio teórico (capítulo 6 del manual):** en clasificación la capa
de salida necesita un nodo por clase con activación **softmax**, que
convierte las salidas en una distribución de probabilidad. Las etiquetas
se preparan con **one-hot encoding**, y la pérdida pasa a ser
`categorical_crossentropy`. Como usamos una red densa (no convolucional
todavía, eso es la próxima clase), primero hay que **aplanar** cada imagen
28x28 en un vector de 784 valores.

In [ ]:
from keras.datasets import mnist
from keras.utils import to_categorical
import matplotlib.pyplot as plt

# MNIST viene incluido en Keras: no requiere descarga manual ni credenciales.
(X_train, y_train), (X_test, y_test) = mnist.load_data()
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

# Visualizamos la primera imagen del set de entrenamiento.
plt.imshow(X_train[0], cmap='gray')
plt.title(f"Etiqueta real: {y_train[0]}")
plt.show()

In [ ]:
# Aplanamos cada imagen 28x28 en un vector de 784 valores (28*28), porque
# una red Dense espera vectores, no matrices.
num_pixels = X_train.shape[1] * X_train.shape[2]
X_train = X_train.reshape(X_train.shape[0], num_pixels).astype('float32')
X_test = X_test.reshape(X_test.shape[0], num_pixels).astype('float32')

# Normalizamos los píxeles de [0, 255] a [0, 1].
X_train = X_train / 255
X_test = X_test / 255

# One-hot encoding de las etiquetas: cada dígito pasa a ser un vector de
# 10 posiciones con un único 1 en la posición correcta.
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)
num_classes = y_test.shape[1]
print("Número de clases:", num_classes)

In [ ]:
def classification_model():
    model = Sequential()
    model.add(Input(shape=(num_pixels,)))
    model.add(Dense(num_pixels, activation='relu'))
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  # una probabilidad por dígito

    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

clf_model = classification_model()
clf_model.summary()

In [ ]:
# Entrenamos usando el conjunto de test como validación en cada época.
# NOTA: con 10 épocas y el dataset completo, esto puede tardar varios
# minutos en un Colab con CPU. Si necesitas resultados más rápidos para
# proyectar en clase, reduce epochs a 3.
clf_history = clf_model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, verbose=2)

scores = clf_model.evaluate(X_test, y_test, verbose=0)
print('Precisión: {:.4f} \n Error: {:.4f}'.format(scores[1], 1 - scores[1]))

### Guardar y recargar el modelo

Keras permite guardar un modelo entrenado en un único archivo `.keras` y
recuperarlo después sin tener que reentrenar desde cero.

In [ ]:
clf_model.save('classification_model.keras')

pretrained_model = keras.saving.load_model('classification_model.keras')
print("Modelo recargado correctamente.")

## Resumen de la clase

- Con Keras, cualquier modelo de deep learning sigue el patrón: preparar
  datos → `Sequential` → `compile` → `fit`.
- **Regresión**: salida de 1 nodo sin activación, pérdida
  `mean_squared_error`.
- **Clasificación**: salida con un nodo por clase y activación `softmax`,
  etiquetas en one-hot, pérdida `categorical_crossentropy`.
- Una red densa necesita que las imágenes se aplanen en vectores antes de
  poder procesarlas; en la próxima clase vas a ver cómo las redes
  convolucionales evitan ese paso y aprovechan la estructura espacial de
  la imagen.

**Sigue con:**
- Capítulos del manual: [`05_regresion_keras.md`](../manual/05_regresion_keras.md) y [`06_clasificacion_keras.md`](../manual/06_clasificacion_keras.md)
- Notebook de práctica de esta clase: [`clase03_practica_regresion_clasificacion_keras.ipynb`](../notebooks_practica/clase03_practica_regresion_clasificacion_keras.ipynb)